# FedAvg by dataset size (baseline)

Aggregates client LoRA adapters (MBPP, HumanEval, DS1000) using **size-weighted** Federated Averaging:
- Weights: \(\alpha_k = n_k / \sum_j n_j\) where \(n_k\) = number of rows in `final_dataset_v2.csv` for dataset \(k\).
- Output: merged adapter at `aggregated_adapters/fedavg_size/` (compatible with existing Verify notebooks).

## 1. Paths and client adapter discovery

In [ ]:
# !pip install -q pandas safetensors torch

In [1]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path

try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

BASE_DIR = Path(NOTEBOOK_DIR)
CSV_PATH = BASE_DIR / "final_dataset_v2.csv"
OUTPUT_DIR = BASE_DIR / "aggregated_adapters" / "fedavg_size"

# Client adapter folders (dataset_name -> path to folder containing adapter_model.safetensors)
CLIENT_CONFIG = {
    "mbpp": BASE_DIR / "MBPP" / "lora_adapters_mbpp",
    "humaneval": BASE_DIR / "HUMANEVAL" / "lora_adapters_humaneval",
    "ds1000": BASE_DIR / "DS1000" / "lora_adapters_ds1000",
}

adapters_found = {k: v for k, v in CLIENT_CONFIG.items() if (v / "adapter_model.safetensors").exists()}
missing = [k for k in CLIENT_CONFIG if k not in adapters_found]
if missing:
    print("Skipping clients (no adapter_model.safetensors):", missing)
print("Clients to aggregate:", list(adapters_found.keys()))

Clients to aggregate: ['mbpp', 'humaneval', 'ds1000']


## 2. Compute size-based weights from final_dataset_v2.csv

In [2]:
df = pd.read_csv(CSV_PATH)
if "dataset" not in df.columns:
    raise ValueError("final_dataset_v2.csv must have a 'dataset' column")

# Normalize dataset names to match CLIENT_CONFIG keys (lowercase)
df["_client"] = df["dataset"].astype(str).str.strip().str.lower()
counts = df["_client"].value_counts()

# Only include clients we have adapters for
n_k = {k: int(counts.get(k, 0)) for k in adapters_found}
total = sum(n_k.values())
if total == 0:
    raise ValueError("No rows in CSV for any of the adapter clients")
alpha = {k: n_k[k] / total for k in adapters_found}

print("Sample counts (n_k):", n_k)
print("Weights (alpha_k):", alpha)

Sample counts (n_k): {'mbpp': 327, 'humaneval': 164, 'ds1000': 1000}
Weights (alpha_k): {'mbpp': 0.2193158953722334, 'humaneval': 0.10999329309188464, 'ds1000': 0.670690811535882}


## 3. Load adapter weights and compute weighted average

In [3]:
from safetensors.torch import load_file, save_file
import torch

state_dicts = {}
keys_ref = None
for name, path in adapters_found.items():
    p = path / "adapter_model.safetensors"
    state_dicts[name] = load_file(str(p))
    if keys_ref is None:
        keys_ref = set(state_dicts[name].keys())
    else:
        if set(state_dicts[name].keys()) != keys_ref:
            raise ValueError(f"Adapter keys differ: {name} vs reference")

print("Loaded", len(state_dicts), "adapters; number of keys:", len(keys_ref))

# Weighted average: theta_global = sum_k alpha_k * theta_k
merged = {}
for key in keys_ref:
    merged[key] = sum(alpha[name] * state_dicts[name][key].float() for name in adapters_found)

print("Merged state dict computed.")

Loaded 3 adapters; number of keys: 504
Merged state dict computed.


## 4. Save merged adapter and config

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_file(merged, str(OUTPUT_DIR / "adapter_model.safetensors"))

# Copy adapter_config.json from first client
first_client_path = adapters_found[list(adapters_found.keys())[0]]
config_src = first_client_path / "adapter_config.json"
if config_src.exists():
    shutil.copy2(str(config_src), str(OUTPUT_DIR / "adapter_config.json"))

# Optionally copy tokenizer/template for convenience
for f in ["tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "README.md"]:
    src = first_client_path / f
    if src.exists():
        shutil.copy2(str(src), str(OUTPUT_DIR / f))

print("Saved to:", OUTPUT_DIR)

Saved to: d:\Desktop\MIT\CODES\FOR GIT HUB\FYP-26-OG\FED-CONS-FINAL\aggregated_adapters\fedavg_size


## 5. Smoke load (optional)

In [ ]:
# Quick check that the merged adapter loads with PEFT
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(base_id, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
print("Smoke load OK. Merged adapter is ready for verification notebooks.")